# Libraries

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Path

In [3]:
BASE_PATH = Path(".")
NB3_PATH = BASE_PATH / "nb3"
OUT_PATH = BASE_PATH / "nb4" 

OUT_PATH.mkdir(parents=True, exist_ok=True)

NODES_CSV = NB3_PATH / "nodes.csv"
EDGES_CSV = NB3_PATH / "edges.csv"
REVIEW_FEAT = NB3_PATH / "review_node_features.npy"

# Load Data
This snippet of code will
1. Loads
- nodes.csv — node table with node_id, node_type, and features.
- edges.csv — edge list with src, dst, edge_type.
- review_node_features.npy — review feature matrix from NB2.

2. Splits nodes into
- user_nodes
- prod_nodes (product)
- review_nodes

3. Builds local index mappings:
- usr2idx, prd2idx, rev2idx

4. Checks that the number of review nodes matches the number of feature rows.

## Loads

In [4]:
nodes = pd.read_csv(NODES_CSV)
edges = pd.read_csv(EDGES_CSV)
review_emb = np.load(REVIEW_FEAT)

In [5]:
print("nodes shape:", nodes.shape)
print("edges shape:", edges.shape)
print("review_emb shape:", review_emb.shape)

nodes shape: (13109, 16)
edges shape: (29512, 3)
review_emb shape: (7378, 777)


## Splitting Nodes

In [6]:
user_nodes   = nodes[nodes["node_type"] == "user"].copy()
prod_nodes   = nodes[nodes["node_type"] == "product"].copy()
review_nodes = nodes[nodes["node_type"] == "review"].copy()

print("num users:", len(user_nodes))
print("num products:", len(prod_nodes))
print("num reviews:", len(review_nodes))

num users: 2755
num products: 2976
num reviews: 7378


## Local Index Mapping

In [ ]:
user_ids   = user_nodes["node_id"].astype(str).tolist()
product_ids = prod_nodes["node_id"].astype(str).tolist()
review_ids  = review_nodes["node_id"].astype(str).tolist()

usr2idx = {u: i for i, u in enumerate(user_ids)}
prd2idx = {a: i for i, a in enumerate(product_ids)}
rev2idx = {r: i for i, r in enumerate(review_ids)}

In [9]:
# Sanity checkkkkkkkkkkkkkkks
assert len(review_ids) == review_emb.shape[0], (
    f"Mismatch: {len(review_ids)} review nodes, "
    f"{review_emb.shape[0]} review_emb rows"
)

# Build HeteroData (Node Features + Edge Index)

Create a HeteroData graph:

- data["review"].x = review feature matrix (review_emb)
- data["user"].x = user stats features
- data["product"].x= item stats features

Build heterogeneous edge indices using the edge types from NB2:

- "user-writes-review" --> (user, "writes", review)
- "review-about-item" --> (review, "about", product)
- "review-written-by-user" --> (review, "written_by", user)
- "item-has-review" --> (product, "has_review", review)

Map string node IDs --> integer indices:

- Use usr2idx, prd2idx, rev2idx.

Move the HeteroData object to the target device

In [10]:
data = HeteroData()

## Node Features

### Review Node Features

In [11]:
data["review"].x = torch.tensor(review_emb, dtype=torch.float32, device=device)

### User Node Features

In [12]:
user_feat_cols = [c for c in user_nodes.columns if c.startswith("user_")]
if user_feat_cols:
    user_feats = user_nodes[user_feat_cols].fillna(0.0).to_numpy()
    data["user"].x = torch.tensor(user_feats, dtype=torch.float32, device=device)
else:
    data["user"].x = torch.zeros((len(user_ids), 1), dtype=torch.float32, device=device)

### Product Node Features

In [13]:
item_feat_cols = [c for c in prod_nodes.columns if c.startswith("item_")]
if item_feat_cols:
    item_feats = prod_nodes[item_feat_cols].fillna(0.0).to_numpy()
    data["product"].x = torch.tensor(item_feats, dtype=torch.float32, device=device)
else:
    data["product"].x = torch.zeros((len(product_ids), 1), dtype=torch.float32, device=device)

## Edges

In [15]:
print("Unique edge types in edges.csv:", edges["edge_type"].unique())

Unique edge types in edges.csv: ['user-writes-review' 'review-about-item' 'review-written-by-user'
 'item-has-review']


### User --> Review

In [ ]:
ur = edges[edges["edge_type"] == "user-writes-review"].copy()

ur["src"] = ur["src"].astype(str)
ur["dst"] = ur["dst"].astype(str)

In [ ]:
u_src_idx = [usr2idx[u] for u in ur["src"]]
u_dst_idx = [rev2idx[r] for r in ur["dst"]]

data["user", "writes", "review"].edge_index = torch.tensor(
    [u_src_idx, u_dst_idx], dtype=torch.long, device=device
)

### Review --> Product 

In [18]:
ri = edges[edges["edge_type"] == "review-about-item"].copy()

ri["src"] = ri["src"].astype(str)
ri["dst"] = ri["dst"].astype(str)

In [19]:
r_src_idx = [rev2idx[r] for r in ri["src"]]
r_dst_idx = [prd2idx[p] for p in ri["dst"]]

data["review", "about", "product"].edge_index = torch.tensor(
    [r_src_idx, r_dst_idx], dtype=torch.long, device=device
)

### Review --> User (Reverse)

In [20]:
ru = edges[edges["edge_type"] == "review-written-by-user"].copy()

ru["src"] = ru["src"].astype(str)
ru["dst"] = ru["dst"].astype(str)

In [21]:
r2_src_idx = [rev2idx[r] for r in ru["src"]]
r2_dst_idx = [usr2idx[u] for u in ru["dst"]]

data["review", "written_by", "user"].edge_index = torch.tensor(
    [r2_src_idx, r2_dst_idx], dtype=torch.long, device=device
)

### Product --> Review

In [22]:
ir = edges[edges["edge_type"] == "item-has-review"].copy()

ir["src"] = ir["src"].astype(str)
ir["dst"] = ir["dst"].astype(str)

In [24]:
i_src_idx = [prd2idx[p] for p in ir["src"]]
i_dst_idx = [rev2idx[r] for r in ir["dst"]]

data["product", "has_review", "review"].edge_index = torch.tensor(
    [i_src_idx, i_dst_idx], dtype=torch.long, device=device
)

## Checks

In [25]:
data

HeteroData(
  review={ x=[7378, 777] },
  user={ x=[2755, 9] },
  product={ x=[2976, 5] },
  (user, writes, review)={ edge_index=[2, 7378] },
  (review, about, product)={ edge_index=[2, 7378] },
  (review, written_by, user)={ edge_index=[2, 7378] },
  (product, has_review, review)={ edge_index=[2, 7378] }
)

In [26]:
data.node_types

['review', 'user', 'product']

In [27]:
data.edge_types

[('user', 'writes', 'review'),
 ('review', 'about', 'product'),
 ('review', 'written_by', 'user'),
 ('product', 'has_review', 'review')]

# SL-GAD Encoder, Decoder, and InfoNCE

SLGADEncoder
- A heterogeneous GNN encoder based on HeteroConv + SAGEConv.
- Uses per-node-type projection to map raw features of each node type (user/product/review) into a shared hidden space.
- Then applies two layers of message passing over the heterogeneous graph.
- Output: embeddings per node type, especially "review" which we care about for anomaly detection.

ReviewDecoder
- A small MLP that reconstructs original review features from review embeddings.
- This implements the generative / reconstruction part of SL-GAD.
- We use MSE between reconstructed and true features as the reconstruction loss.

info_nce
- Implements the contrastive loss between two augmented views (z1, z2).
- Encourages embeddings of the same review across different augmentations to be close, and others to be far.

## Encoder

In [28]:
class SLGADEncoder(nn.Module):
    """
    Heterogeneous GNN encoder for SL-GAD.
    - Per-node-type linear projection to a shared hidden space
    - Two layers of heterogeneous GraphSAGE convolution
    - Returns embeddings for each node type
    """
    def __init__(self, in_dims, hidden=128, out_dim=64, dropout=0.1):
        super().__init__()
        self.out_dim = out_dim
        self.dropout = nn.Dropout(dropout)

        # Per-type input projection
        self.proj = nn.ModuleDict({
            ntype: nn.Linear(in_dim, hidden)
            for ntype, in_dim in in_dims.items()
        })

        # Heterogeneous GraphSAGE layers
        conv1_dict = {}
        conv2_dict = {}
        for (src_t, rel, dst_t) in data.edge_types:
            conv1_dict[(src_t, rel, dst_t)] = SAGEConv((-1, -1), hidden)
            conv2_dict[(src_t, rel, dst_t)] = SAGEConv((-1, -1), out_dim)

        self.conv1 = HeteroConv(conv1_dict, aggr="sum")
        self.conv2 = HeteroConv(conv2_dict, aggr="sum")

    def forward(self, x_dict, edge_index_dict):
        # Node-type-specific projection + ReLU + optional dropout
        h = {
            nt: self.dropout(F.relu(self.proj[nt](x)))
            for nt, x in x_dict.items()
        }
        # First hetero GraphSAGE layer
        h = self.conv1(h, edge_index_dict)
        h = {nt: self.dropout(F.relu(v)) for nt, v in h.items()}
        # Second hetero GraphSAGE layer
        h = self.conv2(h, edge_index_dict)
        return h

## Decoder

In [30]:
class ReviewDecoder(nn.Module):
    """
    MLP decoder that reconstructs review features from review embeddings.
    Used for the generative (reconstruction) loss in SL-GAD.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, z):
        return self.mlp(z)

## Info NCE

In [31]:
def info_nce(z1, z2, tau=0.5):
    """
    InfoNCE contrastive loss between two embedding views.
    Assumes z1[i] and z2[i] are positives; others are negatives.
    """
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    N = z1.size(0)
    logits = torch.mm(z1, z2.t()) / tau   # [N, N]
    labels = torch.arange(N, device=z1.device)
    return F.cross_entropy(logits, labels)

# Model Initialization and Hyperparameters

1. Computes input dimensions per node type from data[nt].x.
2. Initializes SLGADEncoder and ReviewDecoder.
3. Places both on the target device
4. Sets optimizer and hyperparameters:
- learning rate
- weight decay
- number of epochs
- loss weight coefficients (lambda_con, lambda_struct)

## Input Dimension per node type

In [32]:
in_dims = {nt: data[nt].x.size(1) for nt in data.node_types}
print("Input dims per node type:", in_dims)

Input dims per node type: {'review': 777, 'user': 9, 'product': 5}


## Encoder and Decoder Init

In [33]:
encoder = SLGADEncoder(in_dims=in_dims, hidden=128, out_dim=64, dropout=0.1).to(device)
decoder = ReviewDecoder(
    in_dim=encoder.out_dim,
    out_dim=data["review"].x.size(1)
).to(device)

## Optimizer

In [34]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3,
    weight_decay=1e-5,
)

## Parameters

In [35]:
epochs         = 200
lambda_con     = 0.5   # weight for contrastive loss
lambda_struct  = 0.1   # weight for structural consistency loss
grad_clip_norm = 5.0   # gradient clipping threshold

# Training Loop

This is the core SL-GAD training loop. It does:

1. Generate two augmented views of the graph:
- Node feature dropout (random masks).
- Edge dropout per relation type.

2. Compute contrastive loss:
- Encode both views → z1["review"], z2["review"].
- Compute InfoNCE in both directions: info_nce(z1, z2) and info_nce(z2, z1).

3. Compute reconstruction loss:
- Encode base graph → z_base["review"].
- Decode → x_hat.
- MSE with original review features.

4. Compute structural consistency loss:
- Edge dropout only (on base x).
- Encode → z_struct["review"].
- MSE between z_base and z_struct.

5. Combine all into total SL-GAD loss:

6. Backprop, apply gradient clipping, optimizer step.

In [36]:
# ==========================================
# SL-GAD Training Loop
# ==========================================
print("Start training SL-GAD")

loop = tqdm(range(1, epochs + 1), desc="Training")

for epoch in loop:
    encoder.train()
    decoder.train()

    # Base (unperturbed) node features & edges
    x_base = {nt: data[nt].x for nt in data.node_types}
    e_base = {et: data[et].edge_index for et in data.edge_types}

    # -----------------------------
    # 1) Augmented views for contrastive learning
    # -----------------------------
    x_v1, x_v2 = {}, {}
    for nt in data.node_types:
        x = data[nt].x
        # Feature dropout masks
        m1 = (torch.rand_like(x) > 0.2).float()
        m2 = (torch.rand_like(x) > 0.2).float()
        x_v1[nt] = x * m1
        x_v2[nt] = x * m2

    e_v1, e_v2 = {}, {}
    for et in data.edge_types:
        eidx = data[et].edge_index
        E = eidx.size(1)
        # Edge dropout
        k1 = (torch.rand(E, device=device) > 0.2)
        k2 = (torch.rand(E, device=device) > 0.2)
        e_v1[et] = eidx[:, k1]
        e_v2[et] = eidx[:, k2]

    # Encode augmented views
    z1_dict = encoder(x_v1, e_v1)
    z2_dict = encoder(x_v2, e_v2)
    z1 = z1_dict["review"]
    z2 = z2_dict["review"]

    # Symmetric InfoNCE: z1->z2 and z2->z1
    loss_con = info_nce(z1, z2, tau=0.5) + info_nce(z2, z1, tau=0.5)

    # -----------------------------
    # 2) Reconstruction loss on base graph
    # -----------------------------
    z_base_dict = encoder(x_base, e_base)
    z_base = z_base_dict["review"]
    x_true = data["review"].x
    x_hat  = decoder(z_base)
    loss_rec = F.mse_loss(x_hat, x_true)

    # -----------------------------
    # 3) Structural consistency loss (edge dropout only)
    # -----------------------------
    e_struct = {}
    for et in data.edge_types:
        eidx = data[et].edge_index
        keep = (torch.rand(eidx.size(1), device=device) > 0.3)
        e_struct[et] = eidx[:, keep]

    z_struct_dict = encoder(x_base, e_struct)
    z_struct = z_struct_dict["review"]
    loss_struct = F.mse_loss(z_base, z_struct)

    # -----------------------------
    # 4) Total loss, backprop, optimizer step
    # -----------------------------
    loss = loss_rec + lambda_con * loss_con + lambda_struct * loss_struct

    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping to stabilize training
    torch.nn.utils.clip_grad_norm_(
        list(encoder.parameters()) + list(decoder.parameters()),
        max_norm=grad_clip_norm
    )
    optimizer.step()

    loop.set_postfix(
        total=f"{loss.item():.4f}",
        rec=f"{loss_rec.item():.4f}",
        con=f"{loss_con.item():.4f}",
        struct=f"{loss_struct.item():.4f}",
    )

print("SL-GAD training done.")

Start training SL-GAD


Training: 100%|██████████| 200/200 [00:16<00:00, 12.10it/s, con=14.4260, rec=0.0434, struct=0.0792, total=7.2643]

SL-GAD training done.


# Inference, Anomaly Score Computation, and Saving Outputs

1. Switches the model to evaluation mode.
2. Computes final review embeddings on the base graph.
3. Reconstructs review features and measures reconstruction error.
4. Uses reconstruction MSE per review as the anomaly score.

5. Saves:
- slgad_review_embeddings.npy
- slgad_review_scores.npy
- slgad_review_scores.csv (with review_id + anomaly_score)

6. Optionally re-loads the scores file and prints simple stats.

## Switch to Eval

In [37]:
encoder.eval()
decoder.eval()

ReviewDecoder(
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=777, bias=True)
  )
)

## Compute and reconstruct

In [38]:
with torch.no_grad():
    x_base = {nt: data[nt].x for nt in data.node_types}
    e_base = {et: data[et].edge_index for et in data.edge_types}

    z_all = encoder(x_base, e_base)
    z_rev = z_all["review"]                          # [N_reviews, out_dim]
    x_hat_final = decoder(z_rev)                    # reconstructed features
    x_true_final = data["review"].x                 # original features

    # Reconstruction error as anomaly score
    recon_err = ((x_true_final - x_hat_final) ** 2).mean(dim=1)

## Save

In [39]:
emb_np    = z_rev.detach().cpu().numpy()
scores_np = recon_err.detach().cpu().numpy()

# Save embeddings and scores
np.save(OUT_PATH / "slgad_review_embeddings.npy", emb_np)
np.save(OUT_PATH / "slgad_review_scores.npy", scores_np)

In [40]:
out_df = pd.DataFrame({
    "review_id": review_ids,
    "anomaly_score": scores_np,
})
out_df.to_csv(OUT_PATH / "slgad_review_scores.csv", index=False)

In [41]:
print("Embeddings shape:", emb_np.shape)
print("Scores shape:", scores_np.shape)

Embeddings shape: (7378, 64)
Scores shape: (7378,)


# Sanity Check

In [42]:
tmp_scores = np.load(OUT_PATH / "slgad_review_scores.npy")
print("SL-GAD anomaly score stats:")
print("  min :", tmp_scores.min())
print("  max :", tmp_scores.max())
print("  mean:", tmp_scores.mean())
print("  std :", tmp_scores.std())

SL-GAD anomaly score stats:
  min : 0.007478978
  max : 0.27009994
  mean: 0.04298225
  std : 0.015252296
